In [6]:
from io import BytesIO
import logging
from minio import Minio
from minio.error import S3Error
import numpy as np
import os

from torch_data_loader import get_minio_credentials, get_object_list, get_object_from_minio
import torch

DATASET_ROOT_FOLDER = '/home/keithpij/dlio_benchmark/data/unet3d' 
BUCKET_NAME = os.environ['BUCKET_NAME']

In [2]:
BUCKET_NAME

'unet3d'

In [3]:
t = torch.tensor([1,2,3], dtype=torch.float16)

In [7]:
count = 0
split = 'train'
samples_dir = os.path.join(DATASET_ROOT_FOLDER, split)
for entry in os.listdir(samples_dir):
    sample_file_path = os.path.join(samples_dir, entry)
    sample_object_path = f'{split}/{entry}'
    print(sample_file_path, sample_object_path)
    count += 1
    if count > 10: break

/home/keithpij/dlio_benchmark/data/unet3d/train/img_012_of_168.npz train/img_012_of_168.npz
/home/keithpij/dlio_benchmark/data/unet3d/train/img_077_of_168.npz train/img_077_of_168.npz
/home/keithpij/dlio_benchmark/data/unet3d/train/img_075_of_168.npz train/img_075_of_168.npz
/home/keithpij/dlio_benchmark/data/unet3d/train/img_123_of_168.npz train/img_123_of_168.npz
/home/keithpij/dlio_benchmark/data/unet3d/train/img_035_of_168.npz train/img_035_of_168.npz
/home/keithpij/dlio_benchmark/data/unet3d/train/img_061_of_168.npz train/img_061_of_168.npz
/home/keithpij/dlio_benchmark/data/unet3d/train/img_088_of_168.npz train/img_088_of_168.npz
/home/keithpij/dlio_benchmark/data/unet3d/train/img_023_of_168.npz train/img_023_of_168.npz
/home/keithpij/dlio_benchmark/data/unet3d/train/img_145_of_168.npz train/img_145_of_168.npz
/home/keithpij/dlio_benchmark/data/unet3d/train/img_131_of_168.npz train/img_131_of_168.npz
/home/keithpij/dlio_benchmark/data/unet3d/train/img_078_of_168.npz train/img_078

In [8]:
import numpy as np
data = np.load('/home/keithpij/dlio_benchmark/data/unet3d/train/img_011_of_168.npz')
print(data.files)



['x', 'y']


In [9]:
sample = data['x']
label = data['y']
print(sample.shape, label.shape)
print(sample.dtype, label.dtype)
print(label)

(1000, 1000, 7) (7,)
uint8 int64
[0 0 0 0 0 0 0]


In [19]:
data = get_object_from_minio(BUCKET_NAME, 'train/img_011_of_168.npz')

In [20]:
bytes_io = BytesIO(data)
data = np.load(bytes_io)
print(data.files)
sample = data['x']
label = data['y']
print(sample.shape, label.shape)
print(sample.dtype, label.dtype)
print(label)


['x', 'y']
(12857, 18837, 1) (1,)
uint8 int64
[0]


In [24]:
sample_tensor = torch.tensor(sample[0:10000, 0:10000,:], dtype=torch.uint8)
label_tensor = torch.tensor(label, dtype=torch.int64)
print(sample_tensor.shape, label_tensor.shape)
print(sample.dtype, label.dtype)
print(label)


torch.Size([10000, 10000, 1]) torch.Size([1])
uint8 int64
[0]


In [ ]:
object_list = get_object_list(BUCKET_NAME, prefix='train/')
len(object_list)
print(object_list[:10])